# Recycling Alloy Control in Pure Python
## Five Exact Methods for a Two-Scrap Blending Model

This notebook presents five exact approaches for the recycling alloy control problem in pure Python:

1. Naive enumeration
2. Backtracking with pruning
3. Constraint-driven reduced search
4. Dynamic programming
5. Branch and Bound

We solve the course-control problem where two aluminum scraps A and B must be blended to produce `1000` tons of alloy at minimum cost.

Each method reports:
- the best solution found
- the minimum cost
- the execution time


In [1]:
from __future__ import annotations

from functools import lru_cache, wraps
from itertools import combinations, product
from math import ceil, floor, isclose
from time import perf_counter


In [2]:
EPSILON = 1e-9


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


def dot_product(vector_a, vector_b):
    return sum(a * b for a, b in zip(vector_a, vector_b))


def is_integer_value(value: float) -> bool:
    return isclose(value, round(value), abs_tol=EPSILON)


def relation_holds(lhs: float, relation: str, rhs: float) -> bool:
    if relation == "<=":
        return lhs <= rhs + EPSILON
    if relation == ">=":
        return lhs >= rhs - EPSILON
    if relation == "=":
        return isclose(lhs, rhs, abs_tol=EPSILON)
    raise ValueError(f"Unsupported relation: {relation}")


def objective_value(problem, values):
    return dot_product(problem["objective"], values)


def solution_from_values(problem, values):
    solution = {
        name: int(value)
        for name, value in zip(problem["variable_names"], values)
    }
    solution[problem["objective_key"]] = int(round(objective_value(problem, values)))
    return solution


def is_feasible(problem, values):
    for coefficients, relation, rhs in problem["constraints"]:
        lhs = dot_product(coefficients, values)
        if not relation_holds(lhs, relation, rhs):
            return False
    return True


def compare_values(problem, left_values, right_values):
    if left_values is None and right_values is None:
        return 0
    if left_values is None:
        return -1
    if right_values is None:
        return 1

    left_objective = objective_value(problem, left_values)
    right_objective = objective_value(problem, right_values)

    if problem["sense"] == "min":
        if left_objective < right_objective - EPSILON:
            return 1
        if left_objective > right_objective + EPSILON:
            return -1
    else:
        if left_objective > right_objective + EPSILON:
            return 1
        if left_objective < right_objective - EPSILON:
            return -1

    left_key = tuple(int(value) for value in left_values)
    right_key = tuple(int(value) for value in right_values)
    if left_key < right_key:
        return 1
    if left_key > right_key:
        return -1
    return 0


def choose_better(problem, candidate_values, best_values):
    return candidate_values if compare_values(problem, candidate_values, best_values) > 0 else best_values


In [3]:
def feasible_interval_for_b(problem, scrap_a_tons):
    lower_bound, upper_bound = problem["bounds"][1]

    for coefficients, relation, rhs in problem["constraints"]:
        coefficient_a, coefficient_b = coefficients
        constant = rhs - coefficient_a * scrap_a_tons

        if abs(coefficient_b) < EPSILON:
            if not relation_holds(coefficient_a * scrap_a_tons, relation, rhs):
                return None
            continue

        boundary = constant / coefficient_b

        if relation == "<=":
            if coefficient_b > 0:
                upper_bound = min(upper_bound, floor(boundary + EPSILON))
            else:
                lower_bound = max(lower_bound, ceil(boundary - EPSILON))
        elif relation == ">=":
            if coefficient_b > 0:
                lower_bound = max(lower_bound, ceil(boundary - EPSILON))
            else:
                upper_bound = min(upper_bound, floor(boundary + EPSILON))
        elif relation == "=":
            lower_bound = max(lower_bound, ceil(boundary - EPSILON))
            upper_bound = min(upper_bound, floor(boundary + EPSILON))
        else:
            raise ValueError(f"Unsupported relation: {relation}")

        if lower_bound > upper_bound:
            return None

    return int(lower_bound), int(upper_bound)


def best_candidate_for_fixed_a(problem, scrap_a_tons):
    interval = feasible_interval_for_b(problem, scrap_a_tons)
    if interval is None:
        return None

    b_lower, b_upper = interval
    coefficient_b = problem["objective"][1]

    if problem["sense"] == "min":
        scrap_b_tons = b_lower if coefficient_b >= 0 else b_upper
    else:
        scrap_b_tons = b_upper if coefficient_b >= 0 else b_lower

    candidate = (int(scrap_a_tons), int(scrap_b_tons))
    return candidate if is_feasible(problem, candidate) else None


def preferred_b_range(problem, lower_bound, upper_bound):
    coefficient_b = problem["objective"][1]
    should_descend = (
        (problem["sense"] == "max" and coefficient_b >= 0)
        or (problem["sense"] == "min" and coefficient_b < 0)
    )
    if should_descend:
        return range(upper_bound, lower_bound - 1, -1)
    return range(lower_bound, upper_bound + 1)


@timed
def solve_alloy_naive():
    best_values = None
    a_lower, a_upper = ALLOY_PROBLEM["bounds"][0]
    b_lower, b_upper = ALLOY_PROBLEM["bounds"][1]

    for values in product(range(a_lower, a_upper + 1), range(b_lower, b_upper + 1)):
        if is_feasible(ALLOY_PROBLEM, values):
            best_values = choose_better(ALLOY_PROBLEM, values, best_values)

    return solution_from_values(ALLOY_PROBLEM, best_values)


@timed
def solve_alloy_backtracking():
    best_values = None
    a_lower, a_upper = ALLOY_PROBLEM["bounds"][0]

    def backtrack(index, current_values):
        nonlocal best_values

        if index == 0:
            for scrap_a_tons in range(a_lower, a_upper + 1):
                current_values[0] = scrap_a_tons
                optimistic_values = best_candidate_for_fixed_a(ALLOY_PROBLEM, scrap_a_tons)
                if optimistic_values is None:
                    continue
                if best_values is not None and compare_values(ALLOY_PROBLEM, optimistic_values, best_values) <= 0:
                    continue
                backtrack(1, current_values)
            return

        interval = feasible_interval_for_b(ALLOY_PROBLEM, current_values[0])
        if interval is None:
            return

        b_lower, b_upper = interval
        for scrap_b_tons in preferred_b_range(ALLOY_PROBLEM, b_lower, b_upper):
            current_values[1] = scrap_b_tons
            candidate = tuple(current_values)
            if is_feasible(ALLOY_PROBLEM, candidate):
                best_values = choose_better(ALLOY_PROBLEM, candidate, best_values)

    backtrack(0, [0, 0])
    return solution_from_values(ALLOY_PROBLEM, best_values)


@timed
def solve_alloy_reduced_search():
    best_values = None
    a_lower, a_upper = ALLOY_PROBLEM["bounds"][0]

    for scrap_a_tons in range(a_lower, a_upper + 1):
        candidate = best_candidate_for_fixed_a(ALLOY_PROBLEM, scrap_a_tons)
        if candidate is not None:
            best_values = choose_better(ALLOY_PROBLEM, candidate, best_values)

    return solution_from_values(ALLOY_PROBLEM, best_values)


@timed
def solve_alloy_dp():
    a_lower, a_upper = ALLOY_PROBLEM["bounds"][0]
    best_suffix = {a_upper + 1: None}

    for scrap_a_tons in range(a_upper, a_lower - 1, -1):
        current = best_candidate_for_fixed_a(ALLOY_PROBLEM, scrap_a_tons)
        future = best_suffix[scrap_a_tons + 1]
        best_suffix[scrap_a_tons] = choose_better(ALLOY_PROBLEM, current, future)

    return solution_from_values(ALLOY_PROBLEM, best_suffix[a_lower])


In [4]:
def solve_linear_system(matrix, rhs):
    size = len(rhs)
    augmented = [row[:] + [rhs_value] for row, rhs_value in zip(matrix, rhs)]

    for col in range(size):
        pivot_row = max(range(col, size), key=lambda row: abs(augmented[row][col]))
        if abs(augmented[pivot_row][col]) < EPSILON:
            return None

        augmented[col], augmented[pivot_row] = augmented[pivot_row], augmented[col]
        pivot = augmented[col][col]

        for j in range(col, size + 1):
            augmented[col][j] /= pivot

        for row in range(size):
            if row == col:
                continue
            factor = augmented[row][col]
            for j in range(col, size + 1):
                augmented[row][j] -= factor * augmented[col][j]

    return [augmented[row][size] for row in range(size)]


def to_leq_constraints(problem):
    constraints = []
    for coefficients, relation, rhs in problem["constraints"]:
        if relation == "<=":
            constraints.append((list(coefficients), rhs))
        elif relation == ">=":
            constraints.append(([-coefficient for coefficient in coefficients], -rhs))
        elif relation == "=":
            constraints.append((list(coefficients), rhs))
            constraints.append(([-coefficient for coefficient in coefficients], -rhs))
        else:
            raise ValueError(f"Unsupported relation: {relation}")
    return constraints


def solve_lp_relaxation_by_vertices(objective, constraints, bounds, sense):
    number_of_variables = len(objective)
    candidate_equalities = []

    for coefficients, rhs in constraints:
        candidate_equalities.append((coefficients[:], rhs))

    for index, (lower_bound, upper_bound) in enumerate(bounds):
        upper_coefficients = [0.0] * number_of_variables
        upper_coefficients[index] = 1.0
        candidate_equalities.append((upper_coefficients, float(upper_bound)))

        lower_coefficients = [0.0] * number_of_variables
        lower_coefficients[index] = -1.0
        candidate_equalities.append((lower_coefficients, -float(lower_bound)))

    best_solution = None
    best_value = float("inf") if sense == "min" else float("-inf")

    for equality_group in combinations(candidate_equalities, number_of_variables):
        matrix = [list(coefficients) for coefficients, _ in equality_group]
        rhs = [rhs_value for _, rhs_value in equality_group]
        candidate = solve_linear_system(matrix, rhs)

        if candidate is None:
            continue

        if any(
            value < lower_bound - EPSILON or value > upper_bound + EPSILON
            for value, (lower_bound, upper_bound) in zip(candidate, bounds)
        ):
            continue

        if any(dot_product(coefficients, candidate) > rhs_value + EPSILON for coefficients, rhs_value in constraints):
            continue

        candidate_value = dot_product(objective, candidate)
        if sense == "min":
            if candidate_value < best_value - EPSILON:
                best_solution = candidate
                best_value = candidate_value
        else:
            if candidate_value > best_value + EPSILON:
                best_solution = candidate
                best_value = candidate_value

    return best_solution, best_value


@timed
def solve_alloy_branch_and_bound():
    transformed_constraints = to_leq_constraints(ALLOY_PROBLEM)
    root_bounds = [(float(lower), float(upper)) for lower, upper in ALLOY_PROBLEM["bounds"]]
    best_values = None
    best_objective = float("inf")
    stack = [root_bounds]

    while stack:
        bounds = stack.pop()
        lp_solution, lp_value = solve_lp_relaxation_by_vertices(
            objective=ALLOY_PROBLEM["objective"],
            constraints=transformed_constraints,
            bounds=bounds,
            sense=ALLOY_PROBLEM["sense"],
        )

        if lp_solution is None:
            continue

        if best_values is not None and lp_value >= best_objective - EPSILON:
            continue

        fractional_index = None
        for index, value in enumerate(lp_solution):
            if not is_integer_value(value):
                fractional_index = index
                break

        if fractional_index is None:
            integer_values = tuple(int(round(value)) for value in lp_solution)
            if is_feasible(ALLOY_PROBLEM, integer_values):
                best_values = choose_better(ALLOY_PROBLEM, integer_values, best_values)
                best_objective = objective_value(ALLOY_PROBLEM, best_values)
            continue

        fractional_value = lp_solution[fractional_index]
        lower_branch_upper = floor(fractional_value)
        upper_branch_lower = ceil(fractional_value)

        left_bounds = [tuple(bound) for bound in bounds]
        if left_bounds[fractional_index][0] <= lower_branch_upper:
            left_bounds[fractional_index] = (left_bounds[fractional_index][0], float(lower_branch_upper))
            stack.append(left_bounds)

        right_bounds = [tuple(bound) for bound in bounds]
        if upper_branch_lower <= right_bounds[fractional_index][1]:
            right_bounds[fractional_index] = (float(upper_branch_lower), right_bounds[fractional_index][1])
            stack.append(right_bounds)

    return solution_from_values(ALLOY_PROBLEM, best_values)


## Problem: Course Control Alloy Blend

Statement:

A recycling center uses two aluminum scraps, `A` and `B`, to produce a special alloy.

- Scrap `A` contains `6%` aluminum, `3%` silicon, and `4%` carbon.
- Scrap `B` contains `3%` aluminum, `6%` silicon, and `3%` carbon.
- The cost per ton is `$100` for scrap `A` and `$80` for scrap `B`.

The alloy specifications require:
- aluminum content between `3%` and `6%`
- silicon content between `3%` and `5%`
- carbon content between `3%` and `7%`

Determine the optimal mix of scraps needed to produce `1000` tons of the alloy.

Modeling note for this notebook:
- to keep the same exact-methods structure used in the course notebooks, we model the decision in integer tons


In [5]:
ALLOY_PROBLEM = {
    "objective": [100, 80],
    "constraints": [
        ([1, 1], "=", 1000),
        ([0.06, 0.03], ">=", 30),
        ([0.06, 0.03], "<=", 60),
        ([0.03, 0.06], ">=", 30),
        ([0.03, 0.06], "<=", 50),
        ([0.04, 0.03], ">=", 30),
        ([0.04, 0.03], "<=", 70),
    ],
    "bounds": [(0, 1000), (0, 1000)],
    "variable_names": ["scrap_a_tons", "scrap_b_tons"],
    "sense": "min",
    "objective_key": "cost",
}

EXPECTED_SOLUTION = {
    "scrap_a_tons": 334,
    "scrap_b_tons": 666,
    "cost": 86680,
}


In [6]:
naive_result, naive_time = solve_alloy_naive()
backtracking_result, backtracking_time = solve_alloy_backtracking()
reduced_result, reduced_time = solve_alloy_reduced_search()
dp_result, dp_time = solve_alloy_dp()
bb_result, bb_time = solve_alloy_branch_and_bound()

print("=== RECYCLING ALLOY CONTROL RESULTS ===")
print(f"Naive enumeration            -> {naive_result}, time = {naive_time:.8f} seconds")
print(f"Backtracking with pruning    -> {backtracking_result}, time = {backtracking_time:.8f} seconds")
print(f"Constraint-driven reduced search -> {reduced_result}, time = {reduced_time:.8f} seconds")
print(f"Dynamic programming          -> {dp_result}, time = {dp_time:.8f} seconds")
print(f"Branch and Bound             -> {bb_result}, time = {bb_time:.8f} seconds")

assert naive_result == EXPECTED_SOLUTION
assert backtracking_result == naive_result
assert reduced_result == naive_result
assert dp_result == naive_result
assert bb_result == naive_result
print("All five exact methods agree with the expected optimal blend.")


=== RECYCLING ALLOY CONTROL RESULTS ===
Naive enumeration            -> {'scrap_a_tons': 334, 'scrap_b_tons': 666, 'cost': 86680}, time = 0.40025179 seconds
Backtracking with pruning    -> {'scrap_a_tons': 334, 'scrap_b_tons': 666, 'cost': 86680}, time = 0.00411183 seconds
Constraint-driven reduced search -> {'scrap_a_tons': 334, 'scrap_b_tons': 666, 'cost': 86680}, time = 0.00367913 seconds
Dynamic programming          -> {'scrap_a_tons': 334, 'scrap_b_tons': 666, 'cost': 86680}, time = 0.00394454 seconds
Branch and Bound             -> {'scrap_a_tons': 334, 'scrap_b_tons': 666, 'cost': 86680}, time = 0.00082321 seconds
All five exact methods agree with the expected optimal blend.


In [7]:
total_tons = naive_result["scrap_a_tons"] + naive_result["scrap_b_tons"]
aluminum = 0.06 * naive_result["scrap_a_tons"] + 0.03 * naive_result["scrap_b_tons"]
silicon = 0.03 * naive_result["scrap_a_tons"] + 0.06 * naive_result["scrap_b_tons"]
carbon = 0.04 * naive_result["scrap_a_tons"] + 0.03 * naive_result["scrap_b_tons"]

composition_report = {
    "total_tons": total_tons,
    "aluminum_tons": aluminum,
    "silicon_tons": silicon,
    "carbon_tons": carbon,
    "aluminum_pct": 100 * aluminum / total_tons,
    "silicon_pct": 100 * silicon / total_tons,
    "carbon_pct": 100 * carbon / total_tons,
}
composition_report


{'total_tons': 1000,
 'aluminum_tons': 40.019999999999996,
 'silicon_tons': 49.980000000000004,
 'carbon_tons': 33.34,
 'aluminum_pct': 4.002,
 'silicon_pct': 4.998,
 'carbon_pct': 3.3340000000000005}

## Note: Integer vs Continuous Solution

If the decision variables were allowed to take continuous values instead of integer tons, the LP relaxation would produce a slightly cheaper solution.

Using `A + B = 1000`, the cost becomes `100A + 80B = 80000 + 20A`, so the problem minimizes `A`.
The binding restriction is the silicon upper bound:
- `0.03A + 0.06B <= 50`
- replacing `B = 1000 - A` gives `A >= 1000/3`

Therefore the continuous optimum is `A = 1000/3`, `B = 2000/3`, with cost `260000/3`.


In [8]:
continuous_a = 1000 / 3
continuous_b = 2000 / 3
continuous_cost = 260000 / 3

continuous_report = {
    "scrap_a_tons": continuous_a,
    "scrap_b_tons": continuous_b,
    "cost": continuous_cost,
    "gap_vs_integer": EXPECTED_SOLUTION["cost"] - continuous_cost,
}
continuous_report


{'scrap_a_tons': 333.3333333333333,
 'scrap_b_tons': 666.6666666666666,
 'cost': 86666.66666666667,
 'gap_vs_integer': 13.333333333328483}